<a href="https://colab.research.google.com/github/wyattae/cosc-650-applied-llm-systems/blob/WE_1/src/week1/week1_tokenization_starter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 1 (starter): Tokenization Analysis

This is the starter notebook for the Week 1 assignment. It runs as-is on placeholder text so you can see the shape of each step; your job is to replace the placeholders with your own passages and analysis, then commit it to your repository and open a pull request.

Cells marked **TODO (you)** are where you do the work. Everything runs in Jupyter or Google Colab. No GPU, no API key, one dependency: `tiktoken`.

The five parts match the assignment: stand up your repo, run the analysis, evaluate with real figures, find one failure, and submit.

In [1]:
# Setup. In Colab, uncomment the install line on first run.
# !pip install tiktoken
import os, pathlib
os.environ['TIKTOKEN_CACHE_DIR'] = str((pathlib.Path('.') / '.tiktoken_cache').resolve())
os.makedirs(os.environ['TIKTOKEN_CACHE_DIR'], exist_ok=True)

import tiktoken
gpt4  = tiktoken.get_encoding('cl100k_base')   # GPT-4 / GPT-3.5
gpt4o = tiktoken.get_encoding('o200k_base')    # GPT-4o
print('tiktoken', tiktoken.__version__, '- encoders ready (cl100k_base, o200k_base)')

tiktoken 0.14.0 - encoders ready (cl100k_base, o200k_base)


## Part 1: Stand up your repository

Do this once, outside the notebook:

1. Create a public repo (suggested name `cosc-650`).
2. Add a `README.md` a stranger could read (what it is, how it is organized, the tools you use).
3. Add an agent context file that your AI tool reads, with project context and conventions. `AGENTS.md` is the cross-tool convention; `CLAUDE.md` and `GEMINI.md` are tool-specific variants. Use whichever your tool reads.
4. Work on a branch and open a pull request into `main`. You will do this every week.

Then commit this notebook into the repo and keep going.

## Helpers (provided)

Two small functions: count tokens for a string, and show the exact sub-token pieces a word breaks into. The demo uses a line you may recognize.

In [4]:
def count_tokens(text, enc, print_tokens=False):

    tokens = enc.encode(text)
    if print_tokens is True:
      print(tokens)
    return len(tokens)

def show_split(word, enc=gpt4):
    ids = enc.encode(word)
    pieces = [enc.decode([i]) for i in ids]
    print(f'{word!r:18s} -> {len(ids)} token(s): {pieces}')

# demo: some short strings are a single token; capitalized or rarer words fragment
for w in ['Panic', ' towel', '42', 'antidisestablishmentarianism']:
    show_split(w)

'Panic'            -> 2 token(s): ['P', 'anic']
' towel'           -> 1 token(s): [' towel']
'42'               -> 1 token(s): ['42']
'antidisestablishmentarianism' -> 6 token(s): ['ant', 'idis', 'establish', 'ment', 'arian', 'ism']


## Part 2: Your passages

**TODO (you):** replace the two placeholders with your own text. The non-English passage must be at least 100 words, with a faithful English translation. The placeholders below are short Hitchhiker's Guide lines so the notebook runs; swap in your real passages.

In [5]:
# TODO (you): replace both with your own >=100-word passage and its translation.
english_text = "Iran retaliated against US forces on Sunday after the US military struck Iranian rocket launchers, a US official said, marking the first escalation between the two sides in more than a month. Iranian missiles fired toward Jordan were intercepted, the official noted, and no impacts have been reported so far. The attacks followed a US military strike on the Iranian island of Larak, where Iranian forces had been observed preparing to launch rockets toward the Strait of Hormuz. CNN contacted US Central Command for comment on the Iranian attack."
# https://cnnespanol.cnn.com/2026/08/30/mundo/eeuu-ataca-isla-larak-iran-trax
# Translated to English with google translate.
foreign_text = "Irán tomó represalias contra las fuerzas estadounidenses el domingo después de que las Fuerzas Armadas de EE.UU. atacaran lanzacohetes iraníes, dijo un funcionario estadounidense, lo que marcó la primera escalada entre ambas partes en más de un mes. Los misiles iraníes lanzados contra Jordania fueron interceptados, indicó el funcionario, y hasta ahora no se han reportado impactos. Los ataques se produjeron después de que militares de EE.UU. atacaran la isla iraní de Larak, donde observó que fuerzas iraníes se preparaban para lanzar cohetes con minas hacia el estrecho de Ormuz. CNN se puso en contacto con el Comando Central de EE.UU. para solicitar comentarios sobre el ataque iraní."

print('English words:', len(english_text.split()))
print('Foreign words:', len(foreign_text.split()))

def report(label, text):
    print(f'{label:9s} | chars {len(text):4d} | GPT-4 {count_tokens(text, gpt4):4d} | GPT-4o {count_tokens(text, gpt4o):4d}')

report('English', english_text)
report('Foreign', foreign_text)

tax_gpt4  = count_tokens(foreign_text, gpt4)  / count_tokens(english_text, gpt4)
tax_gpt4o = count_tokens(foreign_text, gpt4o) / count_tokens(english_text, gpt4o)
print(f'\nMultilingual tax  GPT-4: {tax_gpt4:.2f}x   GPT-4o: {tax_gpt4o:.2f}x')
# TODO (you): one or two sentences interpreting these numbers for YOUR language pair.

English words: 89
Foreign words: 108
English   | chars  546 | GPT-4  102 | GPT-4o  102
Foreign   | chars  690 | GPT-4  193 | GPT-4o  157

Multilingual tax  GPT-4: 1.89x   GPT-4o: 1.54x


In [15]:
'''
Identify three specific token splits that reveal the tokenizer's English-corpus bias.
For each, show the actual sub-token pieces and contrast with the token count of the English equivalent,
which is not always a single token. To see the shape of a split before you start: the word water is a single token,
while dignity breaks into three pieces (d, ign, ity). Your job is to find where that asymmetry falls hardest between your two languages.
'''
english_word_1 = "United States"
spanish_word_1 = "estadounidense"

english_word_2 = "official"
spanish_word_2 = "funcionario"

english_word_3 = "fired"
spanish_word_3 = "lanzados"

print(f"Word 1.")
show_split(english_word_1)
show_split(spanish_word_1)

print(f"Word 2.")
show_split(english_word_2)
show_split(spanish_word_2)

print(f"Word 3.")
show_split(english_word_3)
show_split(spanish_word_3)

# It's probably worth noting that the english translation seems to be less words. Which does directly impact the token count.
# I was still able to find a few spanish/english pairs that displayed
# token bias, but perhaps analyzing english/spanish is not the best example.

Word 1.
'United States'    -> 2 token(s): ['United', ' States']
'estadounidense'   -> 5 token(s): ['est', 'ad', 'oun', 'id', 'ense']
Word 2.
'official'         -> 1 token(s): ['official']
'funcionario'      -> 2 token(s): ['func', 'ionario']
Word 3.
'fired'            -> 2 token(s): ['f', 'ired']
'lanzados'         -> 3 token(s): ['lan', 'z', 'ados']


"It's probably worth noting that the english translation seems to be less words. Which does directly impact the token count. \nI was still able to find a few spanish/english pairs that displayed\ntoken bias, but perhaps analyzing english/spanish is not the best example."

## Part 3: Evaluate with real figures

Turn the counts into engineering consequences. The skeleton below computes both; keep it pointed at your real passages.

In [17]:
CTX = 128_000
en = count_tokens(english_text, gpt4)
fo = count_tokens(foreign_text, gpt4)
print(f'A {CTX:,}-token window holds about {CTX//en:,} English copies and {CTX//fo:,} foreign copies of your passage.')
print(f'Per-request cost multiplier for the foreign language: {fo/en:.2f}x (billing is per token).')
# TODO (you): state what this means for a product serving users in your chosen language.

# If you were developing an LLM wrapper type application for a spanish audience,
# they would experience increased rate limiting and daily capacity errors as a result
# of their language having more verbose tokenization in comparison to english. In addition to this,
# they would suffer from reduced context in comparison to english translated queries. You see this from the
# results below. 1254 english copies and only 663 spanish copies of the passage can be input into the context window.
# This would ultimately mean that english queries would have more detail and better responses with the additional context.
# Now lets imagine that context window does not matter for our spanish application. This product is still going to be
# more expensive as a result of spanish equivalent passages requiring significantly more tokens per query. In this example,
# we are estimating 1.89x more expensive.

A 128,000-token window holds about 1,254 English copies and 663 foreign copies of your passage.
Per-request cost multiplier for the foreign language: 1.89x (billing is per token).


## Part 4: Bias splits and one failure

**TODO (you):** (a) pick three words where your non-English form fragments far worse than the English equivalent, and show both with `show_split`; (b) find ONE input whose token count defies intuition and explain it. A few failure candidates are demonstrated below to get you started; replace them with your own find and write the explanation plus a mitigation.

In [22]:
# (a) TODO (you): three real bias pairs from your languages.
print('English baselines and spanish translations:')
for w in ['unfortunately:desafortunadamente', 'development:desarrollo', 'environment:medioambiente']:
    words = w.split(":")
    show_split(words[0])
    show_split(words[1])
    print ("")

print('\n(b) failure candidates to explore (replace with your own find):')
show_split('\U0001F680')                 # a rocket emoji
show_split('hello')                       # baseline
show_split(' hello')                      # a leading space changes the tokenization
show_split('https://www.example.com')     # URLs fragment
show_split("🤦🏽‍♂️")                         # Facepalm + skin tone + gender sequence

# TODO (you): explain WHY your chosen case behaves this way, and how you would budget or normalize around it.
# Emojiis are appear to the user as a single character, but they are tokenized on the underlying encoded text rather
# than what appears to be a single symbol. To normalize around this tokenization edgecase, you are incentivized
# to budget around token counts and not character/word counts. You would also want to include additional logic to
# enforce limits around token count instead of word count at the api level. I would also say if the purpose of your application
# does not include use cases like accepting emojiis as input, just sanitize all emojiis from input queries. This is likely
# what should be done for simple company internal LLM applications.

English baselines and spanish translations:
'unfortunately'    -> 2 token(s): ['un', 'fortunately']
'desafortunadamente' -> 4 token(s): ['des', 'afort', 'un', 'adamente']

'development'      -> 1 token(s): ['development']
'desarrollo'       -> 3 token(s): ['des', 'ar', 'rollo']

'environment'      -> 1 token(s): ['environment']
'medioambiente'    -> 4 token(s): ['med', 'io', 'amb', 'iente']


(b) failure candidates to explore (replace with your own find):
'🚀'                -> 3 token(s): ['�', '�', '�']
'hello'            -> 1 token(s): ['hello']
' hello'           -> 1 token(s): [' hello']
'https://www.example.com' -> 5 token(s): ['https', '://', 'www', '.example', '.com']
'🤦🏽\u200d♂️'       -> 11 token(s): ['�', '�', '�', '�', '�', '�', '�', '�', '�', '�', '️']


## Part 5: Submit

Before you open the pull request, check:

- The notebook runs top to bottom on **your** passages, not the placeholders.
- Your three bias splits are shown and explained.
- The failure case has a cause and a mitigation.
- The PR description has a one-paragraph result summary with your headline numbers.
- You linked one issue in your repo logging this as a research note (title, inputs, what you found).

Rubric: repo quality (15), counts from both tokenizers (20), tax computed (15), three bias splits (20), cost and context figures (15), the failure case (10), PR hygiene (5).